# Visualize Huge100K samples

This notebook follows the idea of the sample visualization script, but uses the actual paths from this workspace. It shows how to locate a sample, inspect its `.npy` parameter file, and optionally render a mesh if the required dependencies are available.

In [1]:
from pathlib import Path

# Resolve the dataset root from the repository root or the notebook directory.
candidate_roots = [Path.cwd(), Path.cwd().parent]
DATASET_ROOT = None

for root in candidate_roots:
    candidate = root / "Datasets" / "Huge100K" / "batch 1"
    if candidate.exists():
        DATASET_ROOT = candidate
        break

if DATASET_ROOT is None:
    raise FileNotFoundError("Could not find the Huge100K dataset.")

DATASET_ROOT

PosixPath('/Users/leeon/AvatarVerse/Datasets/Huge100K/batch 1')

In [2]:
# Pick the first image-group folder and inspect its files.
image_groups = sorted([p for p in DATASET_ROOT.iterdir() if p.is_dir() and p.name.startswith("images")])
sample_group = image_groups[0]
sample_group

PosixPath('/Users/leeon/AvatarVerse/Datasets/Huge100K/batch 1/images0')

In [3]:
# Find the sample files inside the selected group.
sample_param_dir = sample_group / "param"
sample_video_dir = sample_group / "videos"

sample_param_files = sorted(sample_param_dir.glob("*.npy"))
sample_image_files = sorted(sample_video_dir.glob("*.jpg"))
sample_video_files = sorted(sample_video_dir.glob("*.mp4"))

sample_param_path = sample_param_files[0]
sample_image_path = sample_image_files[0]
sample_video_path = sample_video_files[0]

print("Parameter file:", sample_param_path)
print("Image file:", sample_image_path)
print("Video file:", sample_video_path)

Parameter file: /Users/leeon/AvatarVerse/Datasets/Huge100K/batch 1/images0/param/Algeria_female_buff_blazers_40~50 years old_279.npy
Image file: /Users/leeon/AvatarVerse/Datasets/Huge100K/batch 1/images0/videos/Algeria_female_buff_blazers_40~50 years old_279.jpg
Video file: /Users/leeon/AvatarVerse/Datasets/Huge100K/batch 1/images0/videos/Algeria_female_buff_blazers_40~50 years old_279.mp4


In [4]:
# Load the parameter file and inspect its contents.
import numpy as np

param = np.load(sample_param_path, allow_pickle=True).item()
print(type(param))
print("Keys:", list(param.keys()))

for key in ["smpl_params", "poses"]:
    if key in param:
        value = param[key]
        print(f"{key}: type={type(value)}, shape={getattr(value, 'shape', None)}")

<class 'dict'>
Keys: ['smpl_params', 'poses']
smpl_params: type=<class 'torch.Tensor'>, shape=torch.Size([189])
poses: type=<class 'list'>, shape=None


In [5]:
# Check whether the rendering dependencies from the original script are available.
import importlib

required_modules = ["torch", "smplx", "trimesh", "pyrender", "imageio"]
available = {}
for name in required_modules:
    try:
        importlib.import_module(name)
        available[name] = "available"
    except Exception as exc:
        available[name] = f"missing: {exc}"

available

{'torch': 'available',
 'smplx': 'available',
 'trimesh': 'available',
 'pyrender': 'available',
 'imageio': 'available'}

In [11]:
# Optional rendering block that mirrors the original script if the dependencies exist.
# If the SMPL-X model directory is not available, this cell will skip rendering.
import os
import importlib

os.environ.setdefault("PYOPENGL_PLATFORM", "osmesa")

SMPLX_MODEL_ROOT = None
candidate_smplx_roots = [
    Path.cwd() / "SMPLX",
    Path.cwd() / "Datasets" / "THuman2.0" / "smplx",
    Path("/Users/leeon/AvatarVerse/SMPLX")
]
for candidate in candidate_smplx_roots:
    if candidate.exists():
        SMPLX_MODEL_ROOT = candidate
        break

print("SMPLX model root:", SMPLX_MODEL_ROOT)

if all(mod in available and available[mod].startswith("available") for mod in ["torch", "smplx", "trimesh", "pyrender", "imageio"]) and SMPLX_MODEL_ROOT is not None:
    import torch
    import smplx
    import trimesh
    import pyrender
    import imageio

    def init_smplx_model():
        body_model = smplx.SMPLX(
            str(SMPLX_MODEL_ROOT),
            gender="neutral",
            create_body_pose=False,
            create_betas=False,
            create_global_orient=False,
            create_transl=False,
            create_expression=True,
            create_jaw_pose=True,
            create_leye_pose=True,
            create_reye_pose=True,
            create_right_hand_pose=False,
            create_left_hand_pose=False,
            use_pca=False,
            num_pca_comps=12,
            num_betas=10,
            flat_hand_mean=False,
        )
        return body_model

    smpl_params = param["smpl_params"].reshape(1, -1)
    scale, transl, global_orient, pose, betas, left_hand_pose, right_hand_pose, jaw_pose, leye_pose, reye_pose, expression = torch.split(
        smpl_params, [1, 3, 3, 63, 10, 45, 45, 3, 3, 3, 10], dim=1
    )

    device = torch.device("cpu")
    model = init_smplx_model().to(device)
    output = model(
        global_orient=global_orient,
        body_pose=pose,
        betas=betas,
        left_hand_pose=left_hand_pose,
        right_hand_pose=right_hand_pose,
        jaw_pose=jaw_pose,
        leye_pose=leye_pose,
        reye_pose=reye_pose,
        expression=expression,
    )
    vertices = output.vertices[0].detach().cpu().numpy()
    faces = model.faces

    mesh = trimesh.Trimesh(vertices, faces)
    mesh_pyrender = pyrender.Mesh.from_trimesh(mesh)

    rendered_images = []
    for idx in range(min(3, len(param["poses"]))):
        scene = pyrender.Scene()
        scene.add(mesh_pyrender)

        camera_params = param["poses"][idx]
        intrinsic_params = camera_params[1]
        extrinsic_params = camera_params[0]

        camera = pyrender.IntrinsicsCamera(fx=intrinsic_params[0], fy=intrinsic_params[1], cx=intrinsic_params[2], cy=intrinsic_params[3])

        extrinsic_params_inv = torch.inverse(extrinsic_params.clone())
        extrinsic_params_inv[:3, 1:3] = -extrinsic_params_inv[:3, 1:3]
        extrinsic_params_inv[3, :3] = 0

        scene.add(camera, pose=extrinsic_params_inv)
        light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=10.0)
        scene.add(light, pose=extrinsic_params_inv)

        renderer = pyrender.OffscreenRenderer(640, 896)
        color, depth = renderer.render(scene)
        rendered_images.append(color)
        renderer.delete()

    print("Rendered images created:", len(rendered_images))
else:
    print("Rendering skipped. Install the required libraries and provide a valid SMPL-X model root to enable it.")

SMPLX model root: None
Rendering skipped. Install the required libraries and provide a valid SMPL-X model root to enable it.
